# EEG · 04 · Generation with frozen SD (Experiment 4)
**Question:** can the EEG-predicted representations guide a reasonable image?

Stable Diffusion, CLIP and the VAE stay **frozen**; only the small token adapter is trained. Needs a GPU and the diffusers weights.

In [1]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config, load_json, get_experiment_paths
from src.data import build_datamodule
from src.generation import generate_images, train_token_adapter
cfg = load_config('configs/EEG/exp04_63_generation.yaml')
dm = build_datamodule(cfg).prepare()
print('experiment:', cfg['experiment']['name'], '| mode:', cfg['generation']['mode'])

project root: c:\Users\xxdia\Documents\TFM\TFM_REPOS_FINALES\tfm_fmri_eeg_diffusion
18:44:08 | INFO    | eeg_datamodule | Loading cached EEG split metadata: data\processed\metadata_eeg_sub-01_63ch.csv
18:44:16 | INFO    | eeg_datamodule | Prepared 1 EEG subject(s) ['sub-01'] | channels=63 | signal={'sub-01': (63, 100)} | agg={'train': 'none', 'val': 'mean', 'test': 'mean'} | images/split: {'train': np.int64(14886), 'val': np.int64(1654), 'test': np.int64(200)}
experiment: exp04_63_eeg_generation | mode: adapter


In [2]:
print(cfg['generation']['mode'])
print(cfg['generation']['adapter_epochs'])

adapter
20


In [3]:
#ejecutar esto si ya habíamos entrenado y queremos cargar sin reentrenar
TRAIN_ADAPTER = False
adapter_ckpt = None

In [3]:
adapter_ckpt = 'outputs/exp04_63_eeg_generation/checkpoints/adapter_best.pt'
print(adapter_ckpt)

outputs/exp04_63_eeg_generation/checkpoints/adapter_best.pt


## (Optional) train the token adapter — the only trainable module

In [3]:
TRAIN_ADAPTER = True
adapter_ckpt = None
if TRAIN_ADAPTER:
    cfg['generation']['adapter_epochs'] = 20
    info = train_token_adapter(cfg, dm, resume='auto')
    adapter_ckpt = info['adapter_checkpoint']
    print('adapter checkpoint:', adapter_ckpt)

c:\Users\xxdia\Documents\TFM\tfm_fmri_diffusion\.tfm_fmri_diffusion_3_11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\xxdia\Documents\TFM\tfm_fmri_diffusion\.tfm_fmri_diffusion_3_11\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


18:15:22 | INFO    | sd | Resumed adapter training from outputs\exp04_63_eeg_generation\checkpoints\adapter_last.pt (epoch 12)
18:15:22 | INFO    | sd | Adapter training: 1 timestep(s)/sample | select best by 'loss' | eval off
18:44:57 | INFO    | sd | [adapter] epoch 12/20 loss 0.15606 (1775s)
19:14:36 | INFO    | sd | [adapter] epoch 13/20 loss 0.15320 (1778s)
19:44:15 | INFO    | sd | [adapter] epoch 14/20 loss 0.15006 (1778s) *
20:13:57 | INFO    | sd | [adapter] epoch 15/20 loss 0.15317 (1779s)
20:43:38 | INFO    | sd | [adapter] epoch 16/20 loss 0.15327 (1781s)
21:13:21 | INFO    | sd | [adapter] epoch 17/20 loss 0.15216 (1783s)
21:43:04 | INFO    | sd | [adapter] epoch 18/20 loss 0.15229 (1782s)
22:12:43 | INFO    | sd | [adapter] epoch 19/20 loss 0.15106 (1778s)
22:12:45 | INFO    | sd | Saved adapter loss curve: outputs\exp04_63_eeg_generation\figures\adapter_loss_curve.png
adapter checkpoint: outputs\exp04_63_eeg_generation\checkpoints\adapter_best.pt


## Generate for the correct condition (and the controls)

In [4]:
decoder = 'outputs/exp03_63_eeg_lowlevel_multitask/checkpoints/best.pt'
outputs = generate_images(cfg, decoder, adapter_checkpoint=adapter_ckpt, split='test')
len(outputs['correct'])

18:44:16 | INFO    | eeg_datamodule | Loading cached EEG split metadata: data\processed\metadata_eeg_sub-01_63ch.csv


18:44:22 | INFO    | eeg_datamodule | Prepared 1 EEG subject(s) ['sub-01'] | channels=63 | signal={'sub-01': (63, 100)} | agg={'train': 'none', 'val': 'mean', 'test': 'mean'} | images/split: {'train': np.int64(14886), 'val': np.int64(1654), 'test': np.int64(200)}


c:\Users\xxdia\Documents\TFM\tfm_fmri_diffusion\.tfm_fmri_diffusion_3_11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 4/4 [00:02<00:00,  1.65it/s]
c:\Users\xxdia\Documents\TFM\tfm_fmri_diffusion\.tfm_fmri_diffusion_3_11\Lib\site-packages\diffusers\pipelines\pipeline_utils.py:2263: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


18:44:33 | INFO    | sd | Loaded token adapter from outputs/exp04_63_eeg_generation/checkpoints/adapter_best.pt
18:46:59 | INFO    | generate | Generated 16 images for condition 'correct'
18:49:20 | INFO    | generate | Generated 16 images for condition 'permuted'
18:51:27 | INFO    | generate | Generated 16 images for condition 'zero'


16

## Real vs generated (correct condition)

In [ ]:
%matplotlib inline

In [ ]:
n = min(6, len(outputs['image_ids']))
fig, axes = plt.subplots(2, n, figsize=(2.4*n, 5))
for j in range(n):
    axes[0,j].imshow(outputs['real'][j]); axes[0,j].axis('off')
    axes[1,j].imshow(outputs['correct'][j]); axes[1,j].axis('off')
axes[0,0].set_ylabel('real'); axes[1,0].set_ylabel('generated')
plt.suptitle('Top: real stimulus — Bottom: brain-guided generation'); plt.tight_layout(); plt.show()

In [ ]:
load_json(get_experiment_paths(cfg, ensure=False).metadata / 'generation_params.json')

**Takeaway:** generation is exploratory here; the quantitative claim comes from the ablation comparison in notebook 05. Note: the adapter training loss is a noisy proxy for generation quality — use notebook 06 to pick the checkpoint by CLIP similarity.